# Financial Market Statistical Analysis

This notebook provides a comprehensive statistical analysis of financial features using the refactored `finance_ml.analytics` module. 
It leverages advanced statistical methods (Bayesian, MCMC, Kalman Filters), interactive dashboards, and performance-optimized operations.


## 1. Setup and Environment Configuration
We import the core analytics modules and configure the visualization environment.


In [ ]:
import logging
import warnings
import os
import pandas as pd
import plotly.express as px

# Configure database connection BEFORE importing analytics modules
# Option 1: Set from environment file or secrets manager
if "DB_URL" not in os.environ:
    # Load from environment_variables.txt if it exists
    env_file = "environment_variables.txt"
    if os.path.exists(env_file):
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ[key.strip()] = value.strip()
    else:
        raise ValueError("DB_URL environment variable not set and environment_variables.txt not found")
else:
    logging.info("DB_URL environment variable already set, skipping loading from file")

# Core analytics imports (consolidated)
from finance_ml.analytics import (
    # Data utilities
    backfill_feature_columns,
    # Feature categories (dynamic loading from DB)
    FEATURE_CATEGORIES,
    compare_registry_with_local,
    _get_fallback_feature_categories,
    # Statistical analysis
    bayesian_category_analysis,
    fit_distributions_by_category,
    hierarchical_mcmc_by_sector,
    kalman_momentum_filter,
    fit_gaussian_copula,
    # Optimized operations
    fast_ruin_probability,
    get_optimization_status,
    # Screening (including new screeners)
    create_enhanced_screener,
    screen_garp_opportunities,
    screen_high_yield_safe_dividends,
    screen_value_opportunities,
    screen_growth_momentum,
    # Feature analytics & dashboards
    PLOTLY_TEMPLATE,
    create_interactive_momentum_dashboard,
    create_interactive_valuation_heatmap,
    create_leverage_liquidity_quadrant,
    bayesian_earnings_beat_model,
    analyze_distress_distribution,
    create_summary_dashboard,
)
# Probability Analytics (Bayesian earnings beat, EPS streaks, model confidence)
from finance_ml.analytics.probability_analytics import (
    EarningsBeatProbabilityModel,
    EPSStreakAnalyzer,
    ModelConfidenceEstimator,
    create_earnings_probability_dashboard,
    create_confidence_calibration_chart,
    create_eps_streak_analysis_chart,
    export_probability_analytics_results,
)
# Category-specific chart functions
from finance_ml.analytics.visualizations.category_charts import (
    # Analyst Sentiment
    create_analyst_sentiment_histogram,
    create_analyst_upside_scatter,
    # Earnings Quality
    create_eps_surprise_histogram,
    create_eps_trajectory_scatter,
    # Growth Metrics
    create_growth_correlation_heatmap,
    create_revenue_vs_eps_growth_scatter,
    # Cash Flow
    create_fcf_margin_yield_scatter,
    create_cash_flow_quality_boxplot,
    # Dividend ReliabilityDividend Reliability
    create_dividend_yield_payout_scatter,
    create_shareholder_yield_histogram,
    # R&D Investment
    create_rnd_intensity_boxplot,
    create_rnd_intensity_growth_scatter,
    create_rnd_per_employee_histogram,
    # Inventory
    create_inventory_days_turnover_scatter,
    # Goodwill & M&A
    create_goodwill_concentration_boxplot,
    create_goodwill_impairment_scatter,
    create_acquisition_activity_histogram,
    # CapEx & Investment
    create_capex_growth_scatter,
    create_investment_efficiency_boxplot,
    create_ma_intensity_histogram,
    # Advanced visualizations (new)
    create_valuation_violin_plot,
    create_quality_risk_radar_chart,
    create_leverage_liquidity_bubble_chart,
)
# Profitability visualizations
from finance_ml.analytics.visualizations.profitability import (
    create_margin_waterfall_chart,
    create_dupont_decomposition_dashboard,
    create_profitability_quadrant,
)
# Technical visualizations
from finance_ml.analytics.visualizations.technical import (
    create_momentum_ribbon_chart,
    create_52w_range_distribution,
)
# Temporal analysis visualizations
from finance_ml.analytics.visualizations.temporal_analysis import (
    create_earnings_calendar_heatmap,
    create_inventory_cycle_analysis,
    create_fcf_trajectory_chart,
    create_dividend_streak_timeline,
)

# Configuration
logging.basicConfig(level=logging.INFO)
warnings.filterwarnings("ignore")
px.defaults.template = PLOTLY_TEMPLATE

# Display optimization and feature category status
opt_status = get_optimization_status()
print(f"JIT Acceleration: {opt_status.get('numba_available')}")
print(f"Feature Categories Loaded: {len(FEATURE_CATEGORIES)}")


## 2. Data Acquisition
Loading feature categories dynamically from the `calculated_features_registry` table (with fallback).
Data is loaded from the `mv_all_stock_features` materialized view.


In [ ]:
# Feature categories are now loaded dynamically from the database at module import
# FEATURE_CATEGORIES is imported from finance_ml.analytics
# Compare with fallback to detect any drift
fallback_categories = _get_fallback_feature_categories()
diff_report = compare_registry_with_local(FEATURE_CATEGORIES, fallback_categories)

print(f"📊 Feature Categories Summary:")
print(f"   Categories loaded: {len(FEATURE_CATEGORIES)}")
print(f"   Total features: {sum(len(v) for v in FEATURE_CATEGORIES.values())}")

if diff_report["features_only_in_db"]:
    print("\n📌 New features in registry (not in fallback):")
    for cat, feats in diff_report["features_only_in_db"].items():
        print(f"   {cat}: {feats}")


In [ ]:
%%sql
SELECT *
FROM public.mv_all_stock_features
WHERE next_earnings >= DATE '2026-01-01' and region = 'Europe'
ORDER BY next_earnings ASC;


In [ ]:
# Normalize SQL result and backfill expected columns using the utility function
if not isinstance(df, pd.DataFrame):
    try:
        df = df.DataFrame()
    except Exception:
        pass

if isinstance(df, pd.DataFrame):
    df = backfill_feature_columns(df)
    print(f"✓ Backfill complete. Columns: {len(df.columns)}")


## 3. Comprehensive Statistical Analysis by Category

This section provides in-depth statistical analysis across all 14 feature categories using Bayesian methods, distribution fitting, and specialized visualizations.


### 3.1 Valuation Ratios
We use Bayesian analysis to estimate true valuation means and visualize valuation metrics across industries.


In [ ]:
val_results = bayesian_category_analysis(df, 'Valuation Ratios', FEATURE_CATEGORIES['Valuation Ratios'])
val_distributions = fit_distributions_by_category(df, 'Valuation Ratios', FEATURE_CATEGORIES['Valuation Ratios'])
create_interactive_valuation_heatmap(df).show()


### 3.2 Momentum & Technical
Applying Kalman filters to smooth momentum signals and visualizing multi-period momentum patterns.


In [ ]:
df_kalman = kalman_momentum_filter(df, momentum_cols=['price_momentum_1y', 'price_momentum_3m'])
momentum_results = bayesian_category_analysis(df, 'Momentum & Technical', FEATURE_CATEGORIES['Momentum & Technical'])
create_interactive_momentum_dashboard(df).show()

In [ ]:
create_52w_range_distribution(df).show()

In [ ]:
create_momentum_ribbon_chart(df).show()


### 3.3 Profitability
Utilizing DuPont decomposition and margin waterfall charts to analyze bottom-line drivers with Bayesian estimation.


In [ ]:
prof_results = bayesian_category_analysis(df, 'Profitability', FEATURE_CATEGORIES['Profitability'])
prof_distributions = fit_distributions_by_category(df, 'Profitability', FEATURE_CATEGORIES['Profitability'])
create_dupont_decomposition_dashboard(df).show()

In [ ]:
create_margin_waterfall_chart(df).show()

In [ ]:
create_profitability_quadrant(df).show()


### 3.4 Quality & Risk
Assessing quality scores and distress risk using Bayesian analysis and tail risk metrics.


In [ ]:
quality_results = bayesian_category_analysis(df, 'Quality & Risk', FEATURE_CATEGORIES['Quality & Risk'])
quality_distributions = fit_distributions_by_category(df, 'Quality & Risk', FEATURE_CATEGORIES['Quality & Risk'])
analyze_distress_distribution(df).show()
df_ruin = fast_ruin_probability(df)


### 3.5 Leverage & Liquidity
Analyzing solvency metrics and balance sheet strength using quadrant analysis and Bayesian estimation.


In [ ]:
leverage_results = bayesian_category_analysis(df, 'Leverage & Liquidity', FEATURE_CATEGORIES['Leverage & Liquidity'])
leverage_distributions = fit_distributions_by_category(df, 'Leverage & Liquidity',
                                                       FEATURE_CATEGORIES['Leverage & Liquidity'])
create_leverage_liquidity_quadrant(df).show()


### 3.6 Analyst Sentiment
Analyzing analyst recommendations, price target upside, and EPS revision momentum.


In [ ]:
sentiment_results = bayesian_category_analysis(df, 'Analyst Sentiment', FEATURE_CATEGORIES['Analyst Sentiment'])
sentiment_distributions = fit_distributions_by_category(df, 'Analyst Sentiment',
                                                        FEATURE_CATEGORIES['Analyst Sentiment'])
create_analyst_sentiment_histogram(df).show()

In [ ]:
create_analyst_upside_scatter(df).show()


### 3.7 Earnings Quality
Evaluating earnings surprises, GAAP adjustments, and earnings trajectory with Bayesian methods.


In [ ]:
earnings_quality_results = bayesian_category_analysis(df, 'Earnings Quality', FEATURE_CATEGORIES['Earnings Quality'])
earnings_quality_distributions = fit_distributions_by_category(df, 'Earnings Quality',
                                                               FEATURE_CATEGORIES['Earnings Quality'])
earnings_beat_probs = bayesian_earnings_beat_model(df)
create_eps_surprise_histogram(df).show()

In [ ]:
create_eps_trajectory_scatter(df).show()


### 3.8 Growth Metrics
Analyzing revenue, EBITDA, EPS, and FCF growth patterns with distribution fitting.


In [ ]:
growth_results = bayesian_category_analysis(df, 'Growth Metrics', FEATURE_CATEGORIES['Growth Metrics'])
growth_distributions = fit_distributions_by_category(df, 'Growth Metrics', FEATURE_CATEGORIES['Growth Metrics'])
create_growth_correlation_heatmap(df, FEATURE_CATEGORIES['Growth Metrics']).show()

In [ ]:
create_revenue_vs_eps_growth_scatter(df).show()


### 3.9 Cash Flow
Analyzing free cash flow metrics, self-funding ratios, and cash flow quality.


In [ ]:
cashflow_results = bayesian_category_analysis(df, 'Cash Flow', FEATURE_CATEGORIES['Cash Flow'])
cashflow_distributions = fit_distributions_by_category(df, 'Cash Flow', FEATURE_CATEGORIES['Cash Flow'])
create_fcf_trajectory_chart(df).show()

In [ ]:
create_fcf_margin_yield_scatter(df).show()

In [ ]:
create_cash_flow_quality_boxplot(df).show()


### 3.10 Dividend Reliability
Evaluating dividend sustainability, payout ratios, and shareholder yield.


In [ ]:
dividend_results = bayesian_category_analysis(df, 'Dividend Reliability', FEATURE_CATEGORIES['Dividend Reliability'])
dividend_distributions = fit_distributions_by_category(df, 'Dividend Reliability', FEATURE_CATEGORIES['Dividend Reliability'])
create_dividend_streak_timeline(df).show()

In [ ]:
create_dividend_yield_payout_scatter(df).show()

In [ ]:
create_shareholder_yield_histogram(df).show()


### 3.11 R&D Investment
Analyzing R&D intensity, growth patterns, and innovation investment efficiency.


In [ ]:
rnd_results = bayesian_category_analysis(df, 'Efficiency Ratios', FEATURE_CATEGORIES['Efficiency Ratios'])
rnd_distributions = fit_distributions_by_category(df, 'Efficiency Ratios', FEATURE_CATEGORIES['Efficiency Ratios'])
create_rnd_intensity_boxplot(df).show()

In [ ]:
create_rnd_intensity_growth_scatter(df).show()

In [ ]:
create_rnd_per_employee_histogram(df).show()


### 3.12 Inventory Temporal
Analyzing inventory cycles, turnover efficiency, and buildup patterns.


In [ ]:
inventory_results = bayesian_category_analysis(df, 'Balance Sheet', FEATURE_CATEGORIES['Balance Sheet'])
inventory_distributions = fit_distributions_by_category(df, 'Balance Sheet',
                                                        FEATURE_CATEGORIES['Balance Sheet'])
create_inventory_cycle_analysis(df).show()

In [ ]:
create_inventory_days_turnover_scatter(df).show()


### 3.13 Goodwill & M&A
Evaluating acquisition activity, goodwill concentration, and impairment risk.


In [ ]:
goodwill_results = bayesian_category_analysis(df, 'Accounting Quality', FEATURE_CATEGORIES['Accounting Quality'])
goodwill_distributions = fit_distributions_by_category(df, 'Accounting Quality', FEATURE_CATEGORIES['Accounting Quality'])
create_goodwill_concentration_boxplot(df).show()

In [ ]:
create_goodwill_impairment_scatter(df).show()

In [ ]:
create_acquisition_activity_histogram(df).show()


### 3.14 CapEx & Investment
Analyzing capital expenditure patterns, investment efficiency, and M&A intensity.


In [ ]:
capex_results = bayesian_category_analysis(df, 'Efficiency Ratios', FEATURE_CATEGORIES['Efficiency Ratios'])
capex_distributions = fit_distributions_by_category(df, 'Efficiency Ratios', FEATURE_CATEGORIES['Efficiency Ratios'])
create_capex_growth_scatter(df).show()

In [ ]:
create_investment_efficiency_boxplot(df).show()

In [ ]:
create_ma_intensity_histogram(df).show()


### 3.15 Probability Analytics: Earnings Beat & EPS Streaks
Advanced probability analysis using Bayesian Beta-Binomial models for earnings beat prediction,
Markov chain-style EPS streak analysis, and model confidence estimation with calibration metrics.


In [ ]:
import numpy as np

# Initialize probability analytics models
beat_model = EarningsBeatProbabilityModel()
streak_analyzer = EPSStreakAnalyzer(mean_reversion_weight=0.3)
confidence_estimator = ModelConfidenceEstimator(n_bins=10)

# Prepare data for analysis - create proxy columns from eps_trajectory_score if needed
df_prob = df.copy()
if 'eps_trajectory_score' in df_prob.columns:
    df_prob['eps_beat_count'] = (df_prob['eps_trajectory_score'].fillna(50) / 100 * 5).astype(int)
    df_prob['eps_total_reports'] = 5

# Compute Bayesian earnings beat probabilities
probability_results = beat_model.analyze_dataframe(
    df_prob,
    beats_col='eps_beat_count',
    total_col='eps_total_reports',
    sector_col='sector' if 'sector' in df_prob.columns else 'industry',
    ticker_col='ticker'
)

print(f"📊 Earnings Beat Probability Analysis")
print(f"   Stocks analyzed: {len(probability_results)}")
if len(probability_results) > 0:
    likely_beat = (probability_results['beat_classification'] == 'likely_beat').sum()
    print(f"   Classified as 'likely beat': {likely_beat} ({likely_beat / len(probability_results) * 100:.1f}%)")
    print(f"   Mean posterior beat probability: {probability_results['posterior_beat_prob'].mean():.1%}")
    print(f"   Mean confidence score: {probability_results['confidence_score'].mean():.2f}")


In [ ]:
# Display top stocks by posterior beat probability
if len(probability_results) > 0:
    top_beat_prob = probability_results.nlargest(50, 'posterior_beat_prob')[
        ['ticker','name', 'historical_beat_rate', 'posterior_beat_prob',
         'ci_90_lower', 'ci_90_upper', 'confidence_score', 'beat_classification']
    ]
    display(top_beat_prob)


In [ ]:
# Earnings Beat Probability Dashboard
create_earnings_probability_dashboard(probability_results).show()


#### EPS Streak Analysis
Analyzing earnings beat/miss streaks with continuation and mean reversion probabilities.


In [ ]:
# Compute EPS streak analysis
streak_results = streak_analyzer.analyze_dataframe(
    df,
    trajectory_col='eps_trajectory_score',
    streak_col='eps_positive_streak' if 'eps_positive_streak' in df.columns else None,
    ticker_col='ticker'
)

print(f"📈 EPS Streak Analysis")
print(f"   Stocks analyzed: {len(streak_results)}")
if len(streak_results) > 0:
    beat_streaks = (streak_results['streak_type'] == 'beat').sum()
    miss_streaks = (streak_results['streak_type'] == 'miss').sum()
    print(f"   On beat streaks: {beat_streaks}")
    print(f"   On miss streaks: {miss_streaks}")
    print(f"   Mean continuation probability: {streak_results['continuation_probability'].mean():.1%}")
    print(f"   Mean reversion probability: {streak_results['mean_reversion_probability'].mean():.1%}")


In [ ]:
# Display stocks with strongest beat streaks
if len(streak_results) > 0:
    strong_streaks = streak_results[streak_results['streak_type'] == 'beat'].nlargest(50, 'current_streak')[
        ['ticker','name', 'current_streak', 'streak_type', 'continuation_probability',
         'mean_reversion_probability', 'expected_next_outcome', 'prediction_confidence']
    ]
    display(strong_streaks)


In [ ]:
# EPS Streak Analysis Chart
create_eps_streak_analysis_chart(streak_results).show()


#### Model Confidence & Calibration
Assessing model reliability using Brier score, calibration error, and AUC-ROC metrics.


In [ ]:
# Compute model confidence metrics (using simulated outcomes for demonstration)
if len(probability_results) > 10:
    np.random.seed(42)
    # Simulate actual outcomes based on posterior probability
    simulated_outcomes = (
            np.random.random(len(probability_results))
            < probability_results['posterior_beat_prob'].values
    ).astype(float)

    confidence_result = confidence_estimator.compute_confidence_metrics(
        predicted_probs=probability_results['posterior_beat_prob'].values,
        actual_outcomes=simulated_outcomes,
        model_name='Bayesian Earnings Beat Model'
    )

    print(f"🎯 Model Confidence Metrics")
    print(f"   Brier Score: {confidence_result.brier_score:.4f} (lower is better, 0=perfect)")
    print(f"   Calibration Error (ECE): {confidence_result.calibration_error:.4f}")
    print(f"   Discrimination (AUC-ROC): {confidence_result.discrimination_auc:.3f}")
    print(f"   Overall Confidence: {confidence_result.overall_confidence:.1f}/100")


In [ ]:
# Model Confidence Calibration Chart
if 'confidence_result' in dir():
    create_confidence_calibration_chart(confidence_result).show()


## 4. Advanced Modeling: Hierarchical Bayes & Copulas
Modeling sector-level dependencies and tail correlations between Valuation and Quality.


In [ ]:
roe_hierarchical = hierarchical_mcmc_by_sector(df, 'roe', sector_col='industry')
copula_fit = fit_gaussian_copula(df, ['p_e_ratio', 'piotroski_f_score'])


## 5. Earnings Calendar Analysis
Visualizing upcoming earnings dates with quality overlay.


In [ ]:
create_earnings_calendar_heatmap(df).show()


## 6. Enhanced Visualizations
New advanced visualizations for comprehensive analysis.


In [ ]:
# Valuation violin plot by industry
create_valuation_violin_plot(df).show()


In [ ]:
# Leverage vs liquidity bubble chart
create_leverage_liquidity_bubble_chart(df).show()


In [ ]:
# Quality radar chart for top stock
if len(df) > 0:
    top_ticker = df.iloc[0]['ticker']
    print(f"Quality Radar for: {top_ticker}")
    create_quality_risk_radar_chart(df, top_ticker).show()


## 7. Stock Screening & Summary
Final ranking and interactive dashboard for the top opportunities using multiple screening strategies.


### 7.1 Enhanced Quality Screener


In [ ]:
# Quality screening with multiple criteria
quality_stocks = create_enhanced_screener(df, min_fscore=7, min_fcf_positive_years=4)
print(f"🏆 Quality Screen: {len(quality_stocks)} stocks found")
if len(quality_stocks) > 0:
    display(quality_stocks[['ticker', 'name', 'sector', 'industry','exchange', 'piotroski_f_score',
                            'distress_risk_score', 'fcf_positive_years']].head(50))


### 7.2 GARP (Growth at Reasonable Price) Screener


In [ ]:
# GARP opportunities
garp_stocks = screen_garp_opportunities(df)
print(f"📈 GARP Screen: {len(garp_stocks)} stocks found")
if len(garp_stocks) > 0:
    cols = ['ticker', 'name','sector', 'industry','country','exchange']
    if 'peg_ratio' in garp_stocks.columns:
        cols.append('peg_ratio')
    if 'eps_yoy_growth' in garp_stocks.columns:
        cols.append('eps_yoy_growth')
    if 'p_e_ratio' in garp_stocks.columns:
        cols.append('p_e_ratio')
    display(garp_stocks[cols].head(50))


### 7.3 High-Yield Safe Dividend Screener


In [ ]:
# Safe high-yield dividends
safe_div_stocks = screen_high_yield_safe_dividends(df)
print(f"💰 Safe High-Yield Dividend Screen: {len(safe_div_stocks)} stocks found")
if len(safe_div_stocks) > 0:
    yield_col = 'dividend_yield_ltm' if 'dividend_yield_ltm' in safe_div_stocks.columns else 'dividend_yield'
    cols = ['ticker', 'name', 'sector', 'industry','country','exchange']
    if 'dividend_payout_ratio' in safe_div_stocks.columns:
        cols.append('dividend_payout_ratio')
    if 'distress_risk_score' in safe_div_stocks.columns:
        cols.append('distress_risk_score')
    display(safe_div_stocks[cols].head(50))


### 7.4 Value Opportunities Screener


In [ ]:
# Value opportunities
value_stocks = screen_value_opportunities(df, max_pe_ratio=20, min_upside_potential=15)
print(f"💎 Value Screen: {len(value_stocks)} stocks found")
if len(value_stocks) > 0:
    cols = ['ticker', 'name','sector', 'industry','country','exchange']
    if 'p_e_ratio' in value_stocks.columns:
        cols.append('p_e_ratio')
    if 'upside_potential' in value_stocks.columns:
        cols.append('upside_potential')
    if 'fcf_yield' in value_stocks.columns:
        cols.append('fcf_yield')
    display(value_stocks[cols].head(50))


### 7.5 Growth Momentum Screener


In [ ]:
# Growth momentum stocks
growth_stocks = screen_growth_momentum(df, min_revenue_growth=5)
print(f"🚀 Growth Momentum Screen: {len(growth_stocks)} stocks found")
if len(growth_stocks) > 0:
    cols = ['ticker', 'name', 'industry']
    if 'revenue_growth_yoy' in growth_stocks.columns:
        cols.append('revenue_growth_yoy')
    if 'eps_yoy_growth' in growth_stocks.columns:
        cols.append('eps_yoy_growth')
    if 'long_term_trend_score' in growth_stocks.columns:
        cols.append('long_term_trend_score')
    display(growth_stocks[cols].head(15))


### 7.6 Summary Dashboard


In [ ]:
create_summary_dashboard(df).show()


### 7.7 Export Probability Analytics Results
Export earnings beat probability analysis, EPS streak analysis, and model confidence metrics to CSV files.


In [ ]:
from pathlib import Path

output_dir = Path('outputs/analytics')
output_dir.mkdir(parents=True, exist_ok=True)

# Export probability analytics results
if len(probability_results) > 0 and len(streak_results) > 0:
    export_paths = export_probability_analytics_results(
        probability_df=probability_results,
        streak_df=streak_results,
        output_dir=output_dir,
        confidence_result=confidence_result if 'confidence_result' in dir() else None
    )
    print("📁 Exported Probability Analytics Results:")
    for name, path in export_paths.items():
        print(f"   ✓ {name}: {path}")


## 8. Analysis Summary


In [ ]:
# Optimization Summary
opt_status = get_optimization_status()
print("=" * 60)
print("📊 ANALYSIS COMPLETE")
print("=" * 60)
print(f"\n🔧 Environment:")
print(f"   JIT Acceleration: {opt_status.get('numba_available')}")
print(f"   Feature Categories: {len(FEATURE_CATEGORIES)}")
print(f"   Total Stocks Analyzed: {len(df)}")

print(f"\n📈 Probability Analytics Summary:")
if len(probability_results) > 0:
    likely_beat = (probability_results['beat_classification'] == 'likely_beat').sum()
    print(f"   Earnings Beat Analysis: {len(probability_results)} stocks")
    print(f"   Likely Beat Classification: {likely_beat} stocks ({likely_beat / len(probability_results) * 100:.1f}%)")
    print(f"   Mean Posterior Beat Prob: {probability_results['posterior_beat_prob'].mean():.1%}")
if len(streak_results) > 0:
    print(f"   EPS Streak Analysis: {len(streak_results)} stocks")
    print(f"   Beat Streaks: {(streak_results['streak_type'] == 'beat').sum()}")
    print(f"   Miss Streaks: {(streak_results['streak_type'] == 'miss').sum()}")
if 'confidence_result' in dir():
    print(f"   Model Confidence Score: {confidence_result.overall_confidence:.1f}/100")

print(f"\n📁 Output Directory: outputs/analytics/")
